In [ ]:
# -*- coding: utf-8 -*-
"""
Created on Sun Dec 29 17:26:52 2024

@author: user
"""
import time
start = time.time()
# =============================================================================
# 
import os
import netCDF4 as nc
import pandas as pd
import numpy as np
import numpy.ma as ma
import xarray as xr
import matplotlib as mpl
import matplotlib.cm as cm
import matplotlib.pyplot as plt
import matplotlib.colors as colors
import matplotlib.style as mplstyle
import matplotlib.dates as mdates
import cartopy.crs as ccrs
import cartopy.feature as cfeat
import cartopy.crs as ccrs
import cartopy.io.img_tiles as cimgt
from cartopy.io.shapereader import Reader
import cartopy.io.shapereader as shpreader
from cartopy.mpl.ticker import LongitudeFormatter,LatitudeFormatter
from cartopy.mpl.gridliner import LONGITUDE_FORMATTER, LATITUDE_FORMATTER
import netCDF4 as nc
from netCDF4 import Dataset
import shapefile
import matplotlib.dates as mdates
# =============================================================================
import sys
import os
# 将目标模块所在的路径添加到 sys.path 中
sys.path.append("/home/yfdong/data/work/LF-SAM/code/Library")

from PreprocessVar import (
    get_MONTH_abbr, get_pentad, Meiyu_configs,
    get_letter, preprocess_var, read_var, 
    Read_VarsFromVarsDict, experiments_types,
    load_geodata, SubsetDomainConfigure ,create_domain_mask,get_letter, UTC_Convert_to_LST
)
import datetime
csvDataPath = f"/raid61/yfdong/data/work/SMP/output/dst/csv/WaterBalance"

In [ ]:
import pandas as pd
# 文件路径
START_YEAR = 2000
END_YEAR = 2024
Years = np.arange(START_YEAR,END_YEAR+1)

def preprocess_WB_Df(Df, START_DATE, END_DATE):
    dataframe = Df.copy()
    dataframe['Date'] = pd.to_datetime(dataframe['Date'])
    dataframe = dataframe[(dataframe['Date'] >= START_DATE) & (dataframe['Date'] <= END_DATE)]
    dataframe = dataframe.sort_values('Date').reset_index(drop=True)  # 添加排序和重置索引
    dataframe['SouthQ'] = dataframe['SouthQin'] - dataframe['SouthQout']
    dataframe['NorthQ'] = dataframe['NorthQin'] - dataframe['NorthQout']
    dataframe['EastQ'] = dataframe['EastQin'] - dataframe['EastQout']
    dataframe['WestQ'] = dataframe['WestQin'] - dataframe['WestQout']
    
    dataframe['ZQ'] = dataframe['WestQ'] + dataframe['EastQ']
    dataframe['MQ'] = dataframe['SouthQ'] + dataframe['NorthQ']
    
    rename_dict = {
    'RUNSB(mm/day)': 'SBROFF',
    'sfcheadrt(mm/day)': 'SFROFF',
    'UDROFF(mm/day)': 'UDROFF'
}
    # 进行重命名操作
    dataframe = dataframe.rename(columns=rename_dict)
    # 在预处理中就添加时区信息
    dataframe['Date_LST'] = dataframe['Date'] + pd.Timedelta(hours=8)
    dataframe = dataframe.rename(columns={'Qin': 'Fin', 'Qout': 'Fout', 'VQ': 'VADV', 'Q': 'HADV', 'SFROFF': 'SFRF', 'SBROFF': 'SBRF', 'UDROFF': 'UDRF', 'TWBR': 'WBR'})
    # 计算总径流
    dataframe['RF'] = dataframe['SFRF'] + dataframe['SBRF']
    # 计算水汽再循环效率
    dataframe['PRR'] = (dataframe['ET'])/(dataframe['ET'] + dataframe['Fin']) *100 # in percentage
    # 处理为daily数据
    dataframe = dataframe.set_index('Date_LST').resample('D').mean().reset_index()
    return dataframe

Total_Expl_df_total = pd.DataFrame()
Total_Ctrl_df_total = pd.DataFrame()
Total_results = {}
Total_results_climate = {}
RegionNames = ['HHP', 'LPT', 'QDW', 'YHP', 'SEH']
CtrlName, ExpName = 'Ctrl', 'FSM'
for RegionName in RegionNames:
    Total_results[RegionName] = {
        CtrlName: pd.DataFrame(),
        ExpName: pd.DataFrame()
    }
    for Year in Years:
        Expl_df_year = pd.read_csv(os.path.join(csvDataPath, f"{Year}-def_WCSMX-WaterBalance-{RegionName}-3H.csv"))
        Ctrl_df_year = pd.read_csv(os.path.join(csvDataPath, f"{Year}-def_WCTRL-WaterBalance-{RegionName}-3H.csv"))
        StartDate, EndDate = (f'{Year}-06-01',f'{Year}-08-30')  
        # 数据预处理
        Ctrl_df_year = preprocess_WB_Df(Ctrl_df_year, StartDate, EndDate)
        Expl_df_year = preprocess_WB_Df(Expl_df_year, StartDate, EndDate)
        # 合并数据
        Total_results[RegionName][ExpName] = pd.concat([Total_results[RegionName][ExpName], Expl_df_year])
        Total_results[RegionName][CtrlName] = pd.concat([Total_results[RegionName][CtrlName], Ctrl_df_year])


In [ ]:
# 初始化气候态平均字典
Total_results_climate = {}

# 遍历每个区域
for RegionName in RegionNames:
    Total_results_climate[RegionName] = {}
    
    # 遍历每个实验（Ctrl 和 WRF-H）
    for ExpType in [CtrlName, ExpName]:
        # 获取该区域该实验的所有年份数据
        df_combined = Total_results[RegionName][ExpType].copy()
        
        # 从Date_LST列提取时间信息用于分组
        df_combined['Month'] = df_combined['Date_LST'].dt.month
        df_combined['Day'] = df_combined['Date_LST'].dt.day
        df_combined['Hour'] = df_combined['Date_LST'].dt.hour
        
        # 按月、日、小时分组计算气候态平均
        climate_avg = df_combined.groupby(['Month', 'Day', 'Hour']).mean(numeric_only=True).reset_index()
        
        # 重新构建时间序列（使用一个参考年份，比如2000年）
        # 这样可以保持时间序列的连续性，便于后续分析
        reference_year = 2000
        climate_avg['Date_LST'] = pd.to_datetime({
            'year': reference_year,
            'month': climate_avg['Month'],
            'day': climate_avg['Day'],
            'hour': climate_avg['Hour']
        })
        
        # 重新排序列，将时间相关列放在前面
        cols = ['Date_LST', 'Month', 'Day', 'Hour'] + [col for col in climate_avg.columns if col not in ['Date_LST', 'Month', 'Day', 'Hour']]
        climate_avg = climate_avg[cols]
        
        # 存储到气候态字典中
        Total_results_climate[RegionName][ExpType] = climate_avg
        
# ---------------------- 计算WRF-H和WRF-S之间的模拟差异 ---------------------
NumericalColumns = ['PW','HADV', 'VADV', 'Prec', 'ET', 'SouthQ', 'NorthQ', 'EastQ', 'WestQ', 'PRR', 'RF',  'SFRF', 'SBRF', 'TSS',  'WBR']

for RegionName in RegionNames:
        Total_results[RegionName]['Diff'] = Total_results[RegionName][ExpName][NumericalColumns] - Total_results[RegionName][CtrlName][NumericalColumns]
        Total_results_climate[RegionName]['Diff'] = Total_results_climate[RegionName][ExpName][NumericalColumns] - Total_results_climate[RegionName][CtrlName][NumericalColumns] 
        Total_results_climate[RegionName]['Diff']['Date_LST'] = Total_results_climate[RegionName]['Ctrl']['Date_LST']

def add_TickGrid(ax, IFminorOn):
    if IFminorOn:
        ax.minorticks_on()
        # 显示副刻度线
        ax.tick_params(axis="both", which="major", direction="in", width=0.9, length=3, color='#363636')
        ax.tick_params(axis="both", which="minor", direction="in", width=0.55, length=1, color='#363636')
        
        # grid 设置网格线性
        ax.grid(True, which="major", linestyle="--", color="gray", linewidth=0.75, alpha = 0.3)
        ax.grid(True, which="minor", linestyle=":", color="gray", linewidth=0.55, alpha = 0.1)
    else:
        ax.minorticks_on()
        # 显示副刻度线
        ax.tick_params(axis="both", which="major", direction="in", width=0.9, length=5, color='#363636')
        # grid 设置网格线性
        ax.grid(True, which="major", linestyle="--", color="gray", linewidth=0.75, alpha = 0.3)
    pass

def DrawBar(ax, ColumnIndex, cols, subTitle, dataframe, BarYrange, IF_Yticks=True):
    # 计算每列的中位数，并将其填充到 NaN 值中
    bar_labels = list(cols.keys())
    # print(bar_labels)
    MEANdf = dataframe[bar_labels]
    # MEANdf = MEANdf.median(numeric_only=True)
    MEANdf = MEANdf.mean(numeric_only=True)
    counts = MEANdf.tolist()
    # 选择需要绘制的变量
    color_list = [sub_dict['color'] for sub_dict in cols.values()]
    bars = ax.bar(bar_labels, counts, label=bar_labels, color=color_list, edgecolor=color_list, alpha=0.6, linewidth=1.5)
    bars = ax.bar(bar_labels, counts, label=bar_labels, color='none', edgecolor=color_list, alpha=1, linewidth=1.5)
    # 添加数据标签
    for bar in bars:
        height = bar.get_height()
        ax.annotate(f'{height:.2f}',
                    xy=(bar.get_x() + bar.get_width() / 2, height-0.005),
                    xytext=(0, 0),  # 3 points vertical offset
                    textcoords="offset points",
                    ha='center', va='bottom',
                    fontsize=fontsize-0.5,
                    style='italic',
                    
                    bbox=dict(boxstyle='round,pad=0.2',  # 背景框样式：圆角，内边距 0.5
          fc='white',  # 背景颜色：黄色
          alpha=0.5,
          ec='none') # 背景透明度：0.5
                    )
    add_TickGrid(ax, IFminorOn=True)
    ax.set_title(f"({get_letter(ColumnIndex)}) {subTitle}",fontsize=fontsize+1, x=0.03, y=0.97, ha='left')
    ax.set_ylim(BarYrange)
    y_min, y_max = ax.get_ylim()
    y_ticks = np.linspace(y_min, y_max, 5)
    ax.set_yticks(y_ticks)
    # 添加标签
    PRR = np.nanmean(dataframe['PRR'])
    # ax.text(0.05, 0.15, f'PRR: {PRR:.2f}%', transform=ax.transAxes, fontsize=fontsize+1, verticalalignment='center', horizontalalignment='left',
    #         bbox=dict(boxstyle='round,pad=0.2', fc='white', alpha=0.5, ec='none'))
    ax.axhline(0, color="#FF242488", linestyle='--', linewidth=0.8)  # 添加零刻度线

    if not IF_Yticks:
        ax.set_yticklabels([])

    # 先设置固定刻度位置（使用当前刻度位置）
    ax.set_xticks(ax.get_xticks())  # 确保刻度位置固定
    ax.set_yticks(ax.get_yticks())  # 确保刻度位置固定

    # 然后安全设置标签
    ax.set_xticklabels(bar_labels, rotation=45, ha='center', fontsize=fontsize)
    ax.set_yticklabels(ax.get_yticklabels(), fontsize=fontsize)  # 保持字体大小设置
    pass

def calculate_errors(dataframe, columns):
    """计算每组数据的标准差作为误差"""
    errors = {}
    for col in columns:
        errors[col] = dataframe[col].std()
    return errors

subTitle = None
IF_legend = False
IF_Diff = True
PlotYrange = (-4, 4)
fontsize = 6.5
Xinterval = 24*10#Hour
IF_Yticks=True
# 修改部分1：在开始绘图前设置全局rcParam参数
plt.rcParams['axes.edgecolor'] = 'black'  # 边框颜色
plt.rcParams['axes.linewidth'] = 0.2     # 边框线宽
TWBcols = {
    "WBR": {'color': "#FF5724", 'linestyle': '-'}, # 残差项
    "TSS": {'color': "#EED600DC", 'linestyle': '-'}, # 土壤湿度变化项
    'Prec': {'color': '#4C4C4C', 'linestyle': '-'},  
    'ET': {'color': "#8BC34A", 'linestyle': '-'},  
    "SFRF": {'color': "#87DDFF", 'linestyle': '-'},
    "SBRF": {'color': "#6CA0DCDF", 'linestyle': '-'},
}
TWBcols = {
    # ——— 残差项（闭合误差，需要视觉突出）———
    "WBR": {'color': "#B2182B", 'linestyle': '-'},  # 深红 - 水量不平衡残余

    # ——— 储水变化项（土壤储水状态）———
    "TSS":  {'color': "#BF812D", 'linestyle': '-'},  # 赭棕 - 土壤湿度变化（土地色）

    # ——— 输入项（大气水汽来源）———
    'Prec': {'color': "#2166AC", 'linestyle': '-'},  # 深蓝 - 降水（大气输入）

    # ——— 输出项：蒸散发（返回大气）———
    'ET':   {'color': "#1B7837", 'linestyle': '-'},  # 深绿 - 蒸散发（植被/生态）

    # ——— 输出项：径流（液态水离开）———
    "SFRF": {'color': "#92C5DE", 'linestyle': '-'},  # 浅蓝 - 地表径流（快速路径）
    "SBRF": {'color': "#4393C3", 'linestyle': '-'},  # 中蓝 - 基流（慢速路径）
}
TWBcols = {
    # ——— 残差项（闭合误差）——— 警示红，应最醒目
    "WBR":  {'color': "#4C4C4C", 'linestyle': '-'},  # 亮红 - 水量不平衡残余

    # ——— 储水变化项 ——— 土壤棕黄，陆地储水
    "TSS":  {'color': "#BF812D", 'linestyle': '-'},  # 赭棕 - 土壤湿度变化

    # ——— 输入项（大气水汽）——— 紫色，与液态水形成冷暖分隔
    'Prec': {'color': "#6A51A3", 'linestyle': '-'},  # 紫罗兰 - 降水（大气水汽）

    # ——— 输出项：蒸散发 ——— 植被绿，生态过程
    'ET':   {'color': "#1B7837", 'linestyle': '-'},  # 深绿 - 蒸散发（植被/生态）

    # ——— 输出项：径流 ——— 同一蓝色系深浅递进，表征快慢路径
    "SFRF": {'color': "#9ECAE1", 'linestyle': '-'},  # 浅蓝 - 地表径流（快速路径）
    "SBRF": {'color': "#2171B5", 'linestyle': '-'},  # 深蓝 - 基流（慢速路径）
}
subTitle = None
IF_legend = False
IF_Diff = True
PlotYrange = (-4, 4)
fontsize = 6.5
Xinterval = 24*10#Hour
IF_Yticks=True
# 修改部分1：在开始绘图前设置全局rcParam参数
plt.rcParams['axes.edgecolor'] = 'black'  # 边框颜色
plt.rcParams['axes.linewidth'] = 0.2     # 边框线宽
# -----------------------------------------------------
NumRows = 3
NumCols = len(RegionNames)
fig, axes = plt.subplots(nrows=NumRows, ncols=NumCols, figsize=(2.*NumCols*0.9, 1.5*NumRows*0.9), dpi=300,
                        sharex=True,
                        # sharey=True,
                        )#, sharey='row') gridspec_kw={'width_ratios': [5, 5, 5, 4]}
DiffMaxYrange = (-2, 3)
CtrlMaxYrange = (-10, 10)
DrawBar(axes[0, 0], 0, TWBcols, f'HHP ({CtrlName})', Total_results_climate['HHP'][CtrlName], BarYrange = CtrlMaxYrange, IF_Yticks= IF_Yticks)
DrawBar(axes[0, 1], 1,  TWBcols, f'LPT ({CtrlName})', Total_results_climate['LPT'][CtrlName], BarYrange = CtrlMaxYrange, IF_Yticks= False)
DrawBar(axes[0, 2], 2,  TWBcols, f'QDW ({CtrlName})', Total_results_climate['QDW'][CtrlName], BarYrange = CtrlMaxYrange, IF_Yticks= False)
DrawBar(axes[0, 3], 3,  TWBcols, f'YHP ({CtrlName})', Total_results_climate['YHP'][CtrlName], BarYrange = CtrlMaxYrange, IF_Yticks= False)
DrawBar(axes[0, 4], 4,  TWBcols, f'SEH ({CtrlName})', Total_results_climate['SEH'][CtrlName], BarYrange = CtrlMaxYrange, IF_Yticks= False)

DrawBar(axes[1, 0], 5, TWBcols, f'HHP ({ExpName})', Total_results_climate['HHP'][ExpName], BarYrange = CtrlMaxYrange, IF_Yticks= IF_Yticks)
DrawBar(axes[1, 1], 6,  TWBcols, f'LPT ({ExpName})', Total_results_climate['LPT'][ExpName], BarYrange = CtrlMaxYrange, IF_Yticks= False)
DrawBar(axes[1, 2], 7,  TWBcols, f'QDW ({ExpName})', Total_results_climate['QDW'][ExpName], BarYrange = CtrlMaxYrange, IF_Yticks= False)
DrawBar(axes[1, 3], 8,  TWBcols, f'YHP ({ExpName})', Total_results_climate['YHP'][ExpName], BarYrange = CtrlMaxYrange, IF_Yticks= False)
DrawBar(axes[1, 4], 9,  TWBcols, f'SEH ({ExpName})', Total_results_climate['SEH'][ExpName], BarYrange = CtrlMaxYrange, IF_Yticks= False)

DrawBar(axes[2, 0], 10, TWBcols, 'HHP (Diff)', Total_results_climate['HHP']['Diff'], BarYrange=DiffMaxYrange, IF_Yticks=IF_Yticks)
DrawBar(axes[2, 1], 11, TWBcols, 'LPT (Diff)', Total_results_climate['LPT']['Diff'], BarYrange=DiffMaxYrange, IF_Yticks=False)
DrawBar(axes[2, 2], 12, TWBcols, 'QDW (Diff)', Total_results_climate['QDW']['Diff'], BarYrange=DiffMaxYrange, IF_Yticks=False)
DrawBar(axes[2, 3], 13, TWBcols, 'YHP (Diff)', Total_results_climate['YHP']['Diff'], BarYrange=DiffMaxYrange, IF_Yticks=False)
DrawBar(axes[2, 4], 14, TWBcols, 'SEH (Diff)', Total_results_climate['SEH']['Diff'], BarYrange=DiffMaxYrange, IF_Yticks=False)

axes[1, 0].set_ylabel('(mm/day)', fontsize=fontsize+1)
# plt.suptitle(f"{RegionName} Region", fontsize=fontsize+1, y=0.92)
plt.subplots_adjust(wspace=0.05, hspace=0.2)